# Blackberry 3DGS: local-stack comparison on Colab

**Runtime -> GPU**, preferably T4 (compute capability 7.5, like the GTX 1650).
Use **Run all** on a fresh VM, or run sections **1 -> 2 -> 2a -> 2b -> 2c -> 3 -> 4 -> 5**.
Colab's kernel Python version does not matter: training uses a separate **Python 3.12.9**.

| Component | Pin |
|---|---|
| Python | 3.12.9 |
| PyTorch | 2.15.0+cu134 |
| CUDA Toolkit / nvcc | 13.4 / 13.4.92 |
| gsplat | 1.6.0, commit `8c33ba2aa4b1283376b679af397ee373d1f4c80e` (upstream PR #1055 CUDA 13 `isfinite` fix) |
| Ninja | 1.13.2 |
| Parallel compiler jobs | **10** |

This compares Linux/GCC with Windows/MSVC, not identical hardware/toolchains.
**10 compiler jobs may exhaust a 12 GB host**; no automatic reduction is applied.
Each test/training cell reloads its own imports, interpreter, CUDA environment and cache path.
After a kernel restart on the **same VM**, these cells can be rerun without reinstalling.
A replacement VM loses `/content`: rerun setup; Drive checkpoints remain available after mounting.

**Do not resume the diverged `cuda_out_mcmc` run** (means reached ~1e5).
Default: stable strategy, fresh `cuda_out_stable`, MCMC only with lower noise.


## 1. GPU check + clone/update repository
Does not replace Colab's driver, system Python or notebook kernel.
Run this cell again after reopening an older notebook to fetch the latest helper.


In [ ]:
import subprocess
import sys
from pathlib import Path

if sys.platform != 'linux':
    raise RuntimeError('Select a Linux Colab GPU runtime.')
print('Notebook kernel:', sys.version, '| training Python: 3.12.9', flush=True)
for command in (['nvidia-smi'], ['g++', '--version']):
    result = subprocess.run(command, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    print(result.stdout, flush=True)
    result.check_returncode()
repo = Path('/content/splats')
branch = 'feature/colab-train'
commands = ([['git', 'clone', '--branch', branch, '--single-branch',
              'https://github.com/aidazsolt-bot/splats.git', str(repo)]] if not repo.is_dir() else
            [['git', '-C', str(repo), 'fetch', 'origin', branch],
             ['git', '-C', str(repo), 'checkout', branch],
             ['git', '-C', str(repo), 'pull', '--ff-only']])
for command in commands:
    result = subprocess.run(command, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    print(result.stdout, flush=True)
    result.check_returncode()


## 2. Install pinned Python / PyTorch / CUDA / gsplat
Downloads Python 3.12.9 via uv, then installs into `/content/splats-cu134-env`.
gsplat compilation is deferred (`BUILD_NO_CUDA=1`). PyTorch3D is optional in section 6.
Installation output is streamed into this notebook; failures stop execution.


In [ ]:
import runpy
from pathlib import Path
if not Path('/content/splats/colab/runtime.py').is_file():
    raise RuntimeError('Repository/helper missing. Run section 1 in this Colab session.')
runtime = runpy.run_path('/content/splats/colab/runtime.py')
runtime['install']()


## 2a. Verify versions + basic CUDA operation
Checks exact Python/Torch/gsplat commit/Ninja/nvcc versions and an actual GPU operation.
Failures here occur before gsplat compilation. No fallback to a different CUDA/Python stack.


In [ ]:
import runpy
from pathlib import Path
if not Path('/content/splats/colab/runtime.py').is_file():
    raise RuntimeError('Repository/helper missing. Run section 1 in this Colab session.')
runtime = runpy.run_path('/content/splats/colab/runtime.py')
ctx = runtime['load_runtime']()
ctx.code('''
import torch
print('GPU:', torch.cuda.get_device_name(0), 'VRAM GiB:', round(torch.cuda.get_device_properties(0).total_memory / 2**30, 1), flush=True)
print('PyTorch architectures:', torch.cuda.get_arch_list(), flush=True)
x = torch.randn(512, 512, device='cuda')
y = x @ x
assert torch.isfinite(y).all().item()
torch.cuda.synchronize()
print('PASS: basic PyTorch CUDA computation', flush=True)
''')


## 2b. Build/load gsplat (verbose, 10 compiler jobs)
CPU/host RAM work, not a GPU training benchmark. Compiler output is streamed live.
Python reports elapsed time and per-process RSS without `/usr/bin/time`.
RSS is **not combined process-tree memory**; reports cannot run after an OOM kill.
Keep the final compiler error on failure. A successful rerun may reuse cached objects.
No cache or lock is deleted automatically.


In [ ]:
import runpy
from pathlib import Path
if not Path('/content/splats/colab/runtime.py').is_file():
    raise RuntimeError('Repository/helper missing. Run section 1 in this Colab session.')
runtime = runpy.run_path('/content/splats/colab/runtime.py')
ctx = runtime['load_runtime']()
ctx.code('''
import resource
import time
started = time.perf_counter()
print('Starting gsplat build/load...', flush=True)
try:
    from gsplat.cuda._backend import _C
    assert _C is not None, 'gsplat CUDA extension unavailable'
    print('PASS: gsplat build/load', flush=True)
finally:
    print(f'Build/load elapsed: {time.perf_counter() - started:.1f} s', flush=True)
    print(f'Python peak RSS: {resource.getrusage(resource.RUSAGE_SELF).ru_maxrss / 1024:.1f} MiB', flush=True)
    print(f'Largest reaped child peak RSS: {resource.getrusage(resource.RUSAGE_CHILDREN).ru_maxrss / 1024:.1f} MiB (not combined process-tree memory)', flush=True)
''')


## 2c. Tiny gsplat forward/backward GPU test
Renders one Gaussian and checks output shapes and finite gradients.
Can be rerun directly after a kernel restart on the same VM; it restores the same build cache.
If the VM was replaced, run sections 1 and 2 again. This test may build if no compiled extension exists.


In [ ]:
import runpy
from pathlib import Path
if not Path('/content/splats/colab/runtime.py').is_file():
    raise RuntimeError('Repository/helper missing. Run section 1 in this Colab session.')
runtime = runpy.run_path('/content/splats/colab/runtime.py')
ctx = runtime['load_runtime']()
ctx.code('''
import torch
from gsplat import rasterization
print('Starting gsplat forward/backward...', flush=True)
means = torch.tensor([[0., 0., 3.]], device='cuda', requires_grad=True)
quats = torch.tensor([[1., 0., 0., 0.]], device='cuda', requires_grad=True)
scales = torch.full((1, 3), 0.1, device='cuda', requires_grad=True)
opacities = torch.tensor([0.8], device='cuda', requires_grad=True)
colors = torch.tensor([[1., 0.5, 0.2]], device='cuda', requires_grad=True)
viewmats = torch.eye(4, device='cuda')[None]
Ks = torch.tensor([[[32., 0., 16.], [0., 32., 16.], [0., 0., 1.]]], device='cuda')
image, alpha, _ = rasterization(means, quats, scales, opacities, colors, viewmats, Ks,
                                 width=32, height=32, render_mode='RGB')
assert image.shape == (1, 32, 32, 3) and alpha.shape == (1, 32, 32, 1)
assert torch.isfinite(image).all().item() and torch.isfinite(alpha).all().item()
assert alpha.max().item() > 0, 'Gaussian was not rendered'
(image.square().mean() + alpha.mean()).backward()
for name, value in [('means', means), ('quats', quats), ('scales', scales), ('opacities', opacities), ('colors', colors)]:
    assert value.grad is not None and torch.isfinite(value.grad).all().item(), f'Invalid gradient: {name}'
torch.cuda.synchronize()
print('PASS: gsplat forward/backward | peak GPU MiB:', round(torch.cuda.max_memory_allocated() / 2**20, 1), flush=True)
''')


## 3. Mount Drive + dataset / output paths
The Drive mount remains in the notebook kernel (not the isolated training Python).
Expected: `MyDrive/blackberry-splats/dataset/{cameras,images,points3D}.txt` and `images/`.
Missing dataset: download once. `OUT` must not be the old diverged MCMC folder.
Paths are saved on this VM so later cells also work after a kernel restart.


In [ ]:
import runpy
import subprocess
from pathlib import Path
from google.colab import drive

drive.mount('/content/drive')
DATA = '/content/drive/MyDrive/blackberry-splats/dataset'
OUT = '/content/drive/MyDrive/blackberry-splats/cuda_out_stable'
if not Path('/content/splats/colab/runtime.py').is_file():
    raise RuntimeError('Repository/helper missing. Run section 1 in this Colab session.')
runtime = runpy.run_path('/content/splats/colab/runtime.py')
if not (Path(DATA) / 'cameras.txt').is_file():
    print('Downloading missing dataset...', flush=True)
    Path(DATA).mkdir(parents=True, exist_ok=True)
    runtime['run_live'](['wget', '-O', '/content/blackberry.zip',
                         'https://github.com/danybittel/splats/releases/download/berries/blackberry.zip'])
    runtime['run_live'](['unzip', '-q', '-o', '/content/blackberry.zip', '-d', DATA])
runtime['save_paths'](DATA, OUT)
DATA, OUT = runtime['load_paths']()
print('DATA:', DATA, '| OUT:', OUT, flush=True)


## 4. Smoke train
500 steps, default strategy, data-factor 4. Outputs go to `OUT/smoke`.


In [ ]:
import runpy
from pathlib import Path
if not Path('/content/splats/colab/runtime.py').is_file():
    raise RuntimeError('Repository/helper missing. Run section 1 in this Colab session.')
runtime = runpy.run_path('/content/splats/colab/runtime.py')
ctx = runtime['load_runtime']()
DATA, OUT = runtime['load_paths']()
ctx.run(['engines_cuda/run_train.py', '--smoke', '--data', DATA,
         '--out', f'{OUT}/smoke', '--data-factor', '4', '--strategy', 'default'])


## 5. Full train (stable recipe)
Default strategy is recommended. MCMC: use low `NOISE_LR` (e.g. 5e3), not the diverged 5e5 recipe.
T4: start with data-factor 2; L4/A100 can try 1.
After a healthy disconnect: set `RESUME=True`, keeping the same OUT and training settings.


In [ ]:
import runpy
from pathlib import Path
if not Path('/content/splats/colab/runtime.py').is_file():
    raise RuntimeError('Repository/helper missing. Run section 1 in this Colab session.')
runtime = runpy.run_path('/content/splats/colab/runtime.py')
ctx = runtime['load_runtime']()
DATA, OUT = runtime['load_paths']()

RESUME = False
STRATEGY = 'default'
DATA_FACTOR = 2
CAP_MAX = 200000
MAX_STEPS = 30000
NOISE_LR = 5e3
ANTIALIASED = True
cmd = ['engines_cuda/run_train.py', '--data', DATA, '--out', OUT,
       '--strategy', STRATEGY, '--max-steps', str(MAX_STEPS), '--data-factor', str(DATA_FACTOR)]
if STRATEGY == 'mcmc':
    cmd += ['--cap-max', str(CAP_MAX), '--noise-lr', str(NOISE_LR)]
if ANTIALIASED:
    cmd += ['--antialiased']
if RESUME:
    cmd += ['--resume']
ctx.run(cmd)


## 5b. Quick PLY sanity check
Run after the first `splat_*.ply`. Extent should be roughly 20-70, not 1e5+.


In [ ]:
import runpy
from pathlib import Path
if not Path('/content/splats/colab/runtime.py').is_file():
    raise RuntimeError('Repository/helper missing. Run section 1 in this Colab session.')
runtime = runpy.run_path('/content/splats/colab/runtime.py')
ctx = runtime['load_runtime']()
DATA, OUT = runtime['load_paths']()
ctx.code(f'OUT = {OUT!r}\n' + r'''
import numpy as np
from pathlib import Path
ply = sorted(Path(OUT).glob('splat_*.ply'))
assert ply, f'No splat_*.ply yet in {OUT}'
path = ply[-1]
raw = path.read_bytes()
i = raw.find(b'end_header\n')
assert i >= 0, 'PLY header terminator missing'
hdr, body = raw[:i].decode(), raw[i + 11:]
n = int([ln for ln in hdr.splitlines() if ln.startswith('element vertex')][0].split()[-1])
props = [ln.split()[-1] for ln in hdr.splitlines() if ln.startswith('property float')]
arr = np.frombuffer(body, dtype='<f4', count=n * len(props)).reshape(n, len(props))
ix = {p: j for j, p in enumerate(props)}
xyz = arr[:, [ix['x'], ix['y'], ix['z']]]
opa = 1 / (1 + np.exp(-np.clip(arr[:, ix['opacity']], -40, 40)))
extent = xyz.max(0) - xyz.min(0)
print(path.name, 'n', n, 'extent', extent, 'center', xyz.mean(0), flush=True)
print('opacity p50/mean', float(np.median(opa)), float(opa.mean()), flush=True)
assert np.isfinite(xyz).all() and np.isfinite(opa).all(), 'Non-finite splat values'
assert extent.max() <= 500, 'Means likely diverged: stop and change STRATEGY/NOISE_LR/OUT'
print('OK: extent looks COLMAP-scale', flush=True)
''')


## 6. List outputs / optional mesh
Open3D Poisson is installed. PyTorch3D cubify is optional and has a separate source build.
Set `INSTALL_PYTORCH3D=True` only if needed. Mesh export stays opt-in (`EXPORT_MESH`).


In [ ]:
import runpy
from pathlib import Path
if not Path('/content/splats/colab/runtime.py').is_file():
    raise RuntimeError('Repository/helper missing. Run section 1 in this Colab session.')
runtime = runpy.run_path('/content/splats/colab/runtime.py')
ctx = runtime['load_runtime']()
DATA, OUT = runtime['load_paths']()
for path in sorted(Path(OUT).iterdir()):
    print(path.name, flush=True)
INSTALL_PYTORCH3D = False
EXPORT_MESH = False
BACKEND = 'open3d'
if INSTALL_PYTORCH3D:
    ctx.pip(['install', '-c', '/content/splats-cu134-env/comparison-constraints.txt',
             '--no-build-isolation', '-r', 'engines_cuda/requirements-pytorch3d.txt'])
if EXPORT_MESH:
    ctx.run(['engines_cuda/export/mesh_from_ckpt.py', '--ckpt', f'{OUT}/ckpt_30000.pt',
             '--out', f'{OUT}/mesh_from_splat.ply', '--backend', BACKEND])
